# Task 1 — Inconsistencies and duplicates

Cleaning `club_signups.csv` into one row per student per club.

In [2]:
import pandas as pd

pd.set_option("display.max_columns", 30)

raw = pd.read_csv("club_signups.csv")   
signups = raw.copy()                     # every fix happens on this copy

n_start = len(raw)
print("rows at the start:", n_start)

rows at the start: 39


## Part A — Inconsistencies

### 1. Look first.
Shape, dtypes, and `value_counts()` of every categorical column, so the problems are visible before anything is fixed.

In [3]:
print(raw.shape)
print()
print(raw.dtypes)

(39, 9)

signup_id       object
student_id      object
full_name       object
email           object
faculty         object
city            object
club            object
signed_up_at    object
fee_paid        object
dtype: object


In [4]:
for col in ["faculty", "city", "club", "fee_paid"]:
    print(raw[col].value_counts(), "\n")

faculty
Pharmacy                                  7
EMS                                       7
IET                                       5
Management                                4
Information Engineering and Technology    4
iet                                       3
MET                                       3
Engineering and Materials Science         1
Pharma                                    1
Media Engineering and Technology          1
met                                       1
pharmacy                                  1
ems                                       1
Name: count, dtype: int64 

city
Cairo         13
Alexandria    11
Giza           4
Alex           4
cairo          2
giza           2
alexandria     1
CAIRO          1
El Giza        1
Name: count, dtype: int64 

club
Robotics       8
Music          5
Chess          4
football       4
Debating       3
Football       3
Chess Club     3
Soccer         2
music          2
chess          2
debate club    1
Music         

What this shows: `faculty`, `city` and `club` each have many spellings (different case, extra spaces, short forms like `Alex`, full names like `Media Engineering and Technology`, `Soccer` for Football). `fee_paid` is text in nine different spellings, and everything is `object` (text) including `signed_up_at` and `fee_paid`.

### 2. Faculty, club, city
 **normalise mechanically** (strip spaces, lower-case), then an **explicit dictionary** for what is left. Finally **assert** that only the canonical values remain.

In [5]:
FACULTIES = ["MET", "IET", "EMS", "Pharmacy", "Management"]
CLUBS = ["Robotics", "Debate", "Music", "Football", "Chess"]
CITIES = ["Cairo", "Giza", "Alexandria"]

faculty_map = {
    "met": "MET", "media engineering and technology": "MET",
    "iet": "IET", "information engineering and technology": "IET",
    "ems": "EMS", "engineering and materials science": "EMS",
    "pharmacy": "Pharmacy", "pharma": "Pharmacy",
    "management": "Management",
}
club_map = {
    "robotics": "Robotics",
    "debate": "Debate", "debating": "Debate", "debate club": "Debate",
    "music": "Music",
    "football": "Football", "soccer": "Football",
    "chess": "Chess", "chess club": "Chess",
}
city_map = {
    "cairo": "Cairo",
    "giza": "Giza", "el giza": "Giza",
    "alexandria": "Alexandria", "alex": "Alexandria",
}

signups["faculty"] = signups["faculty"].str.strip().str.lower().map(faculty_map)
signups["club"] = signups["club"].str.strip().str.lower().map(club_map)
signups["city"] = signups["city"].str.strip().str.lower().map(city_map)

`.map(dict)` returns NaN for any value that is not in the dictionary, so a spelling I forgot would show up as a missing value. The asserts below fail if that happens, or if anything outside the allowed values remains.

In [6]:
assert signups["faculty"].isin(FACULTIES).all()
assert signups["club"].isin(CLUBS).all()
assert signups["city"].isin(CITIES).all()

for col in ["faculty", "club", "city"]:
    print(signups[col].value_counts(), "\n")

faculty
IET           12
Pharmacy       9
EMS            9
MET            5
Management     4
Name: count, dtype: int64 

club
Football    9
Chess       9
Music       8
Robotics    8
Debate      5
Name: count, dtype: int64 

city
Alexandria    16
Cairo         16
Giza           7
Name: count, dtype: int64 



### 3. Names and emails.
Names: collapse any run of spaces into one, trim the ends, and use one capitalisation (`Title Case`). Emails: trim and lower-case. I also check that `student_id` is already clean (it is the key I will deduplicate on later), and that every `student_id` ends up with exactly one name and one email.

In [7]:
signups["full_name"] = (
    signups["full_name"].str.replace(r"\s+", " ", regex=True).str.strip().str.title()
)
signups["email"] = signups["email"].str.strip().str.lower()

assert signups["student_id"].str.fullmatch(r"\d{2}-\d{4}").all()   # e.g. 58-1234, nothing else
assert (signups.groupby("student_id")["full_name"].nunique() == 1).all()
assert (signups.groupby("student_id")["email"].nunique() == 1).all()

signups[["student_id", "full_name", "email"]].drop_duplicates().sort_values("student_id")

,student_id,full_name,email
27,55-2639,Youssef Fathy,youssef.fathy@student.guc.edu.eg
19,55-2971,Salma Ibrahim,salma.ibrahim@student.guc.edu.eg
14,55-2992,Mohamed Adel,mohamed.adel@student.guc.edu.eg
5,55-9245,Ali Ibrahim,ali.ibrahim@student.guc.edu.eg
26,58-1089,Karim Kamel,karim.kamel@student.guc.edu.eg
12,58-1125,Ziad Saleh,ziad.saleh@student.guc.edu.eg
16,58-8701,Farida Ibrahim,farida.ibrahim@student.guc.edu.eg
17,58-9118,Mariam Kamel,mariam.kamel@student.guc.edu.eg
6,61-2495,Ahmed Adel,ahmed.adel@student.guc.edu.eg
1,61-3846,Ahmed Zaki,ahmed.zaki@student.guc.edu.eg


**Two different students have the same name.** Two `Mohamed Adel`s: different `student_id`, and they even share the same email. So name and email are *not* keys. Everything that follows uses `student_id`.

In [8]:
same_name = signups.groupby("full_name")["student_id"].nunique()
print(same_name[same_name > 1], "\n")

signups[signups["full_name"] == "Mohamed Adel"][["student_id", "full_name", "email"]].drop_duplicates()

full_name
Mohamed Adel    2
Name: student_id, dtype: int64 



,student_id,full_name,email
2,61-4844,Mohamed Adel,mohamed.adel@student.guc.edu.eg
14,55-2992,Mohamed Adel,mohamed.adel@student.guc.edu.eg


### 4. `fee_paid`.
Lower-case and strip, then an explicit dictionary so that it becomes a boolean column.

In [9]:
fee_map = {
    "yes": True, "y": True, "true": True, "1": True,
    "no": False, "n": False, "false": False, "0": False,
}

signups["fee_paid"] = signups["fee_paid"].astype(str).str.strip().str.lower().map(fee_map)
assert signups["fee_paid"].notna().all()
signups["fee_paid"] = signups["fee_paid"].astype(bool)

print(signups["fee_paid"].dtype)
print(signups["fee_paid"].value_counts())

bool
fee_paid
True     27
False    12
Name: count, dtype: int64


### 5. `signed_up_at`
Two formats are mixed: `2026-09-15 13:39` (year first) and `15/09/2026 03:12` (with slashes). 

In [10]:
is_iso = signups["signed_up_at"].str.match(r"\d{4}-\d{2}-\d{2}")
print("year-first rows:", is_iso.sum(), "| slash rows:", (~is_iso).sum())

slash = signups.loc[~is_iso, "signed_up_at"]
first = slash.str.slice(0, 2).astype(int)    # the part before the first "/"
second = slash.str.slice(3, 5).astype(int)   # the part between the slashes
print("first number ranges from", first.min(), "to", first.max())
print("second number ranges from", second.min(), "to", second.max())

year-first rows: 25 | slash rows: 14
first number ranges from 13 to 18
second number ranges from 9 to 9


The first number goes above 12 (so it **cannot** be a month), and the second number is always 9 (September, which matches the year-first rows that all fall in September). So the slash format is **day/month/year**.

I parse each format with its own explicit `format=`, then combine the two. `.where(...)` blanks out the rows of the other format, and `fillna` fills them from the second parse.

In [11]:
year_first = pd.to_datetime(signups["signed_up_at"].where(is_iso), format="%Y-%m-%d %H:%M")
day_first = pd.to_datetime(signups["signed_up_at"].where(~is_iso), format="%d/%m/%Y %H:%M")

signups["signed_up_at"] = year_first.fillna(day_first)

assert signups["signed_up_at"].notna().all()
assert signups["signed_up_at"].dt.year.eq(2026).all() and signups["signed_up_at"].dt.month.eq(9).all()
print(signups["signed_up_at"].dtype)
print(signups["signed_up_at"].min(), "->", signups["signed_up_at"].max())

datetime64[ns]
2026-09-13 14:36:00 -> 2026-09-18 23:36:00


The times have no timezone in the file, so I left them as given.

## Part B — Duplicates

### 1. Exact duplicates.
Rows where **every** column is identical (same `signup_id` too): the same form submission recorded twice. I count them, look at them, remove the extra copies, and count again.

In [12]:
n_exact = signups.duplicated().sum()
print("exact duplicate rows:", n_exact)
signups[signups.duplicated(keep=False)].sort_values("signup_id")

exact duplicate rows: 3


,signup_id,student_id,full_name,email,faculty,city,club,signed_up_at,fee_paid
0,S015,61-8942,Hana Zaki,hana.zaki@student.guc.edu.eg,Pharmacy,Giza,Football,2026-09-15 13:39:00,False
34,S015,61-8942,Hana Zaki,hana.zaki@student.guc.edu.eg,Pharmacy,Giza,Football,2026-09-15 13:39:00,False
15,S022,61-8224,Sara Zaki,sara.zaki@student.guc.edu.eg,EMS,Giza,Music,2026-09-16 01:39:00,True
24,S022,61-8224,Sara Zaki,sara.zaki@student.guc.edu.eg,EMS,Giza,Music,2026-09-16 01:39:00,True
20,S023,58-1125,Ziad Saleh,ziad.saleh@student.guc.edu.eg,IET,Cairo,Chess,2026-09-16 09:29:00,False
33,S023,58-1125,Ziad Saleh,ziad.saleh@student.guc.edu.eg,IET,Cairo,Chess,2026-09-16 09:29:00,False


In [13]:
before = len(signups)
signups = signups.drop_duplicates().reset_index(drop=True)
n_after_exact = len(signups)

assert signups["signup_id"].is_unique       # every signup_id now appears once
print(f"{before} -> {n_after_exact} rows  (removed {before - n_after_exact})")

39 -> 36 rows  (removed 3)


### 2. The same sign up submitted again
Different `signup_id`, same `student_id` and same `club`. Usually the student came back to mark the fee as paid, so the **latest** submission is the true state.

In [14]:
signups = signups.sort_values("signed_up_at").reset_index(drop=True)

assert not signups.duplicated(["student_id", "club", "signed_up_at"]).any()

repeats = signups[signups.duplicated(["student_id", "club"], keep=False)].sort_values(["student_id", "club", "signed_up_at"])
repeats[["signup_id", "student_id", "club", "signed_up_at", "fee_paid"]]

,signup_id,student_id,club,signed_up_at,fee_paid
19,S020,55-9245,Music,2026-09-15 23:07:00,False
23,S033,55-9245,Music,2026-09-16 09:07:00,True
20,S021,61-4844,Debate,2026-09-16 01:15:00,False
30,S035,61-4844,Debate,2026-09-17 11:15:00,True
16,S017,61-7344,Debate,2026-09-15 18:01:00,False
22,S034,61-7344,Debate,2026-09-16 02:01:00,True
34,S032,61-7344,Football,2026-09-17 23:36:00,False
35,S036,61-7344,Football,2026-09-18 23:36:00,True


In [15]:
before = len(signups)
signups = signups.drop_duplicates(["student_id", "club"], keep="last")
n_final = len(signups)
n_same_club = before - n_final

signups = signups.sort_values("signup_id").reset_index(drop=True)
print(f"{before} -> {n_final} rows  (removed {n_same_club})")

36 -> 32 rows  (removed 4)


### 3. Check:

In [16]:
assert not signups.duplicated(["student_id", "club"]).any()
assert signups["student_id"].str.fullmatch(r"\d{2}-\d{4}").all()
print("one row per student per club: OK")

one row per student per club: OK


## Does the order of the steps matter?
What if I had removed duplicates **before** fixing the spellings, working directly on `raw`? Same two duplicate steps, on the untouched data:

In [17]:
raw_exact = raw.duplicated().sum()
raw_no_exact = raw.drop_duplicates()
raw_same_club = raw_no_exact.duplicated(["student_id", "club"]).sum()

print("exact duplicates       : on raw =", raw_exact, "| after cleaning =", n_exact)
print("same student + club    : on raw =", raw_same_club, "| after cleaning =", n_same_club)
print("repeat sign-ups missed by deduplicating first:", n_same_club - raw_same_club)

exact duplicates       : on raw = 3 | after cleaning = 3
same student + club    : on raw = 2 | after cleaning = 4
repeat sign-ups missed by deduplicating first: 2


Exact duplicates are found either way (identical text is identical). But the *same-student-same-club* repeats are not: `Music` vs `music`, `Debate` vs `debate club` are different strings until the spellings are fixed, so deduplicating first misses those repeats and leaves two rows for one student and club.

## Summary, save, and a last check

In [18]:
print("rows at the start             :", n_start)
print("removed as exact duplicates    :", n_exact)
print("removed as same student + club :", n_same_club)
print("rows left                      :", n_final)
assert n_start - n_exact - n_same_club == n_final == len(signups)

signups.to_csv("club_signups_clean.csv", index=False)

assert raw.equals(pd.read_csv("club_signups.csv"))
clean_back = pd.read_csv("club_signups_clean.csv")
assert len(clean_back) == n_final
signups.dtypes

rows at the start             : 39
removed as exact duplicates    : 3
removed as same student + club : 4
rows left                      : 32


signup_id               object
student_id              object
full_name               object
email                   object
faculty                 object
city                    object
club                    object
signed_up_at    datetime64[ns]
fee_paid                  bool
dtype: object